In [1]:
from training import count_parameters, TrainConfig, train_waitk_student, TranslationDataset, load_training_checkpoint
import mlflow
import torch
import json
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKHybridMamba2Adapter
from model_classes import WaitKHybridMamba2MT, SimulHybridMamba2Config

/root/LinearSimultMT/.venv-mamba/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_hybrid_distillation")

from transformers import AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [ ]:
hybrid_config = SimulHybridMamba2Config(
    vocab_size = len(tokenizer),
    d_model = 512,
    num_encoder_layers = 6,
    num_decoder_layers = 6,
    d_state = 128,
    d_conv = 4,
    expand = 2,
    headdim = 64,
    ngroups = 1,
    nhead = 8,
    dim_feedforward = 2048,
    dropout = 0.1,
    max_source_len = 64,
    max_target_len = 64,
    pad_token_id = tokenizer.pad_token_id,
    eos_token_id = tokenizer.eos_token_id,
)

In [ ]:
hybrid_mamba2 = WaitKHybridMamba2MT(hybrid_config)

In [ ]:
count_parameters(hybrid_mamba2)

In [ ]:
train_cfg = TrainConfig(
    epochs=10,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=[3, 5, 7, 9, 11],

    use_kl_loss=False,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=2e-5,
    use_amp=True,
)

In [ ]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [ ]:
train_waitk_student(
    student=hybrid_mamba2,
    train_dataset=dataset,
    model_cfg=hybrid_config,
    train_cfg=train_cfg,
    device="cuda",
    checkpoint_name_prefix="hybrid",
    resume_from_checkpoint="./checkpoints/hybrid_step_12000.pt"
)

In [3]:
hybrid_mamba2 = load_training_checkpoint("./models/hybrid.pt", SimulHybridMamba2Config, WaitKHybridMamba2MT)[0]

In [ ]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

adapter = WaitKHybridMamba2Adapter(
    model=hybrid_mamba2,
    tokenizer=tokenizer,
    name="hybrid_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(
        use_comet=True,
        comet_model_name="Unbabel/wmt22-comet-da",
        comet_batch_size=512,
    ),
)

for k in [3, 5, 7, 9, 11]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63
    )
    
    print(result.metrics)
    
    with open(f"./metrics/hybrid_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/root/LinearSimultMT/.venv-mamba/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


Validating hybrid_wait_k, wait_k=3:   0%|          | 0/303 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/

{'BLEU': 32.23075330505572, 'chrF++': 54.499283690741926, 'TER': 61.479870501962054, 'COMET': 0.8001670514951444, 'AP': 0.6645900322666223, 'AL': 4.475102289445077, 'DAL': 5.225846177727015, 'LAAL': 4.77429856247584, 'ATD_text': 3.174575301953112, 'total_time_sec': 502.0180051690004, 'ms_per_sentence': 1.6225009054943291, 'target_tokens_per_sec': 17849.214386212057, 'source_tokens_per_sec': 15503.9498979321, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 10002.07666015625, 'dataset_fraction': 1.0, 'eval_dataset_size': 309410, 'full_dataset_size': 309410, 'generation_total_time_sec': 457.39686202298617, 'generation_ms_per_sentence': 1.4782872629294017, 'generation_target_tokens_per_sec': 19590.48638936594}


Validating hybrid_wait_k, wait_k=5:   0%|          | 0/303 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1209/1209 [23:19<00:00, 

{'BLEU': 34.451777380053656, 'chrF++': 56.16760712764305, 'TER': 58.27140912530797, 'COMET': 0.82040767441968, 'AP': 0.7305274362183886, 'AL': 6.320220457098759, 'DAL': 7.0995701448791335, 'LAAL': 6.182454476386909, 'ATD_text': 4.451102953246316, 'total_time_sec': 507.2369374600021, 'ms_per_sentence': 1.6393682733589803, 'target_tokens_per_sec': 17484.28859382768, 'source_tokens_per_sec': 15344.43063033781, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 7646.03369140625, 'dataset_fraction': 1.0, 'eval_dataset_size': 309410, 'full_dataset_size': 309410, 'generation_total_time_sec': 460.45712899299906, 'generation_ms_per_sentence': 1.488177916011115, 'generation_target_tokens_per_sec': 19260.592227978825}


Validating hybrid_wait_k, wait_k=7:   0%|          | 0/303 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0:   3%|██████                                                                                                                                                                          | 42/1209 [00:16<07:37, 